In [0]:
from pyspark.sql.functions import *

#fething last_run_time to process only new data instead of full data

last_run_time = spark.sql("select last_run_time from data_engineering.table_metadata.Control_Table where pipeline_name='medallion_architecture_sales_pipeline'").collect()[0][0]

df=spark.table("data_engineering.silver_layer.silver_sales").filter(col("ingestion_time")>last_run_time)

#total saler per city

df_total_sales_per_city= df.groupby("city").agg(sum("sales").alias("total_sales"))

#df_total_sales_per_city.write.format("delta").mode("overwrite").option("overwriteSchema","true").partitionBy("city").saveAsTable("data_engineering.gold_layer.total_sales_per_city")

df_total_sales_per_city.createOrReplaceTempView("df_total_sales_per_city")

display(df_total_sales_per_city)





Merge Logic for incremental load

In [0]:
%sql

Merge into data_engineering.gold_layer.total_sales_per_city as target using df_total_sales_per_city as source
on target.city = source.city
when matched then 
  update set target.total_sales = target.total_sales + source.total_sales
when not matched then insert *



In [0]:
print("Gold merge completed successfully")



Update control table with later run_time_date

In [0]:
%sql

UPDATE data_engineering.table_metadata.Control_Table
SET last_run_time = current_timestamp(),
    status = 'SUCCESS'
WHERE pipeline_name = 'medallion_architecture_sales_pipeline';